In [ ]:
#!/usr/bin/env python3
"""
plot_figures.py: Reproduce Figs.2 and 5 from data files using Matplotlib.
"""
import os
import numpy as np
import matplotlib.pyplot as plt

# Configure LaTeX rendering and style
plt.rcParams.update({
    "text.usetex": True,
    "font.size": 9,
    "mathtext.fontset": "cm",
    "axes.linewidth": 0.75,
    "xtick.major.width": 0.75,
    "ytick.major.width": 0.75,
    "xtick.major.size": 1,
    "ytick.major.size": 1,
    "savefig.dpi": 100,
    "font.family": "serif",
    "text.latex.preamble": r"\usepackage{amsmath}\usepackage{amssymb}"
})


def get_project_root() -> str:
    """
    Locate the directory that contains a 'submission' folder one level above.
    Skips a folder literally named 'data' to prefer the proper project root.
    """
    try:
        base = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base = os.getcwd()
    current = base
    while True:
        # Direct submission folder present
        if os.path.isdir(os.path.join(current, 'submission')) and os.path.basename(current).lower() != 'data':
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    # Fallback: use base
    return base

SCRIPT_DIR = get_project_root()


def load_data(relative_path: str) -> np.ndarray:
    """
    Load a whitespace-delimited text file given by relative_path under project root.
    Tries:
      1) SCRIPT_DIR/relative_path
      2) SCRIPT_DIR/data/relative_path
    Raises FileNotFoundError if neither exists.
    """
    # 1) direct
    path1 = os.path.join(SCRIPT_DIR, relative_path)
    if os.path.isfile(path1):
        return np.loadtxt(path1)
    # 2) under data/
    path2 = os.path.join(SCRIPT_DIR, 'data', relative_path)
    if os.path.isfile(path2):
        return np.loadtxt(path2)
    raise FileNotFoundError(f"Data file not found: {path1} or {path2}")


def make_fig_5():
    data1 = load_data(os.path.join('submission', 'double_ocupany_bos_fermi_couple0.1.txt'))
    data3 = load_data(os.path.join('submission', 'double_ocupany_bos_fermi_couple5.txt'))
    data4 = load_data(os.path.join('submission', 'total_boson_num0.1.txt'))
    data6 = load_data(os.path.join('submission', 'total_boson_num5.txt'))

    fig = plt.figure(figsize=(3.29, 0.75 * 6))
    gs = fig.add_gridspec(3, 1)
    ax1 = fig.add_subplot(gs[0, 0])
    ax3 = fig.add_subplot(gs[1, 0], sharex=ax1)
    ax4 = fig.add_subplot(gs[2, 0], sharex=ax1)

    ax1.plot(data1[:, 0], data1[:, 1], marker='o', linestyle='-', lw=1.25, markersize=0,
             color='black', label=r'$j=1$')
    ax1.plot(data1[:, 0], data1[:, 2], marker='^', linestyle='--', lw=1.25, markersize=0,
             color='darkgoldenrod', label=r'$j=2$')
    ax1.plot(data1[:, 0], data1[:, 1] + data1[:, 2], marker='^', linestyle=':', lw=0.75,
             markersize=0, color='blue', label=r'total')

    ax3.plot(data3[:, 0], data3[:, 1], marker='o', linestyle='-', lw=1.25, markersize=0,
             color='black', label=r'$j=1$')
    ax3.plot(data3[:, 0], data3[:, 2], marker='^', linestyle='--', lw=1.25, markersize=0,
             color='darkgoldenrod', label=r'$j=2$')
    ax3.plot(data3[:, 0], data3[:, 1] + data3[:, 2], marker='^', linestyle=':', lw=0.75,
             markersize=0, color='blue', label=r'total')

    ax4.plot(data4[:, 0], data4[:, 1] + data4[:, 2], marker='o', linestyle='--', lw=1.5,
             markersize=0, color='red', label=r'$g/k=0.1$')
    ax4.plot(data6[:, 0], data6[:, 1] + data6[:, 2], marker='^', linestyle='-', lw=0.75,
             markersize=0, color='deepskyblue', label=r'$g/k=5.0$')

    ax4.set_xlabel(r'$t\: k$')
    for ax in (ax1, ax3): ax.set_xlim(0,20); ax.set_ylim(0,1.4); ax.set_ylabel(r'$\langle \hat{n}_{j, \uparrow}\hat{n}_{j, \downarrow} \rangle$')
    ax4.set_xlim(0,20); ax4.set_ylim(-0.2,6.75); ax4.set_ylabel(r'$\langle \hat{n}_{ph} \rangle$')

    ax1.set_yticks([0,0.5,1.0],[r'0',r'0.5',r'1.0']); ax3.set_yticks([0,0.5,1.0],[r'0',r'0.5',r'1.0'])
    ax1.tick_params(labelbottom=False); ax3.tick_params(labelbottom=False)

    ax3.legend(labelspacing=0, frameon=False, borderpad=-0.4); ax4.legend(labelspacing=0, frameon=False, borderpad=-0.4)
    ax1.text(0.5,1.2,r'a.'); ax3.text(0.5,1.2,r'b.'); ax4.text(0.5,5.8,r'c.')
    ax1.text(10,1.2,r'$g/k=0.1$',ha='center'); ax3.text(10,1.2,r'$g/k=5.0$',ha='center')

    fig.subplots_adjust(hspace=0); plt.tight_layout(); plt.show()


def make_fig_2():
    data_g5   = load_data(os.path.join('submission','fidelity_5.txt'))
    data_g01  = load_data(os.path.join('submission','fidelity_0.1.txt'))
    data_noisy= load_data(os.path.join('submission','noisy_fidelity_Juan.txt'))

    fig=plt.figure(figsize=(6.5,1.5)); gs=fig.add_gridspec(1,3,wspace=0.4)
    ax1,ax2,ax3=[fig.add_subplot(gs[0,i]) for i in range(3)]

    for ax,data,label in ((ax1,data_g01,r'$g=0.1k$'),(ax2,data_noisy,r'$g=k$')):
        for col,mark in zip(range(1,5),('o','*','^','H')):
            ax.plot(data[:,0],data[:,col],marker=mark,lw=0.5,markersize=3)
        ax.set_xlim(data_g5[:,0].min(),data_g5[:,0].max());ax.set_ylim(-0.05,1.15)
        ax.set_ylabel(r'Fidelity',labelpad=0);ax.set_xlabel(r'$t\: k$')
        ax.text(2.5,1.05,label)
    ax1.text(0.25,1.05,r'a.');ax2.text(0.25,1.05,r'b.');ax1.legend(('DAQC(I)','DAQC(N)','Dig(I)','Dig(N)'),labelspacing=0,frameon=True,loc='best',handlelength=1.5)

    for col,mark in zip(range(1,5),('o','*','^','H')):
        ax3.plot(data_g5[:,0],data_g5[:,col],marker=mark,lw=0.5,markersize=3)
    ax3.set_xlim(data_g5[:,0].min(),data_g5[:,0].max());ax3.set_ylim(-0.05,1.15)
    ax3.set_ylabel(r'Fidelity',labelpad=0);ax3.set_xlabel(r'$t\: k$')
    ax3.text(2.5,1.05,r'$g=5k$');ax3.text(0.25,1.05,r'c.');ax3.set_yticklabels([])

    plt.tight_layout();plt.show()


def main():
    print(f"Using project root: {SCRIPT_DIR}")
    make_fig_2(); make_fig_5(); input('Press any key to exit.')


if __name__=='__main__':
    main()